In [1]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file, 'r', encoding='utf-8') as file:
        content = file.read()
    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc.md")

In [2]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk)
    return embedding.tolist()
embeddings = [embed_chunk(chunk) for chunk in chunks]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chroma_db")
chroma_collection = chroma_client.get_or_create_collection(name="my_collection")

def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    ids = [str(i) for i in range(len(chunks))]
    chroma_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )

save_embeddings(chunks, embeddings)

In [4]:
def retrieve(query: str, top_k: int) -> List[str]:
    '''召回'''
    query_embedding = embed_chunk(query)
    res = chroma_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return res['documents'][0]

query = "哆啦A梦使用的3个秘密道具分别是什么?"

retrieved_chunks = retrieve(query, 5)


In [10]:
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    '''重排'''
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)
    chunk_with_score_list = [(chunk, score) for chunk, score in zip(retrieved_chunks, scores)]
    chunk_with_score_list.sort(key=lambda pair: pair[1], reverse=True)
    return [chunk for chunk, _ in chunk_with_score_list][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 1)
print(reranked_chunks)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

['三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。']


In [13]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
googlt_client = genai.Client()

def generate(query: str, chunks: List[str]):
    prompt = f"""你是一位知识助手, 请根据用户的问题和下列片段生成准确的回答.

用户问题: {query}

相关片段:
{"\n\n".join(chunks)}

请鲫鱼上述内容作答,不要编造信息."""

    print(f"{prompt}\n\n---\n")

    response = googlt_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text
answer = generate(query, reranked_chunks)
print(answer)

你是一位知识助手, 请根据用户的问题和下列片段生成准确的回答.

用户问题: 哆啦A梦使用的3个秘密道具分别是什么?

相关片段:
三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。

请鲫鱼上述内容作答,不要编造信息.

---

根据您提供的片段，哆啦A梦使用的三件秘密道具分别是：

1.  **复制斗篷**：可以临时赋予超级战力。
2.  **时间停止手表**：能暂停时间五秒。
3.  **精神与时光屋便携版**：可在一分钟中完成一年修行。
